# 有机锂中间体稳定性预测：全局 Arrhenius 方法**目标**: 输入 SMILES + 温度 → 预测 t_max, t½ → 推荐 flash / flow / batch 反应器**方法**: 全局 Arrhenius 拟合（所有温度同时拟合 5 参数）+ 类别特异描述符模型**流程**:1. 读取原始数据2. 数据可视化3. 逐温度动力学拟合 (对比用)4. **全局 Arrhenius 拟合** (核心新方法)5. 描述符准备6. 类别特异描述符筛选7. 端到端验证8. 预测工具

## Step 1: 读取原始数据

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom scipy.optimize import curve_fitfrom sklearn.linear_model import LinearRegressionfrom sklearn.model_selection import LeaveOneOutfrom rdkit import Chemfrom rdkit.Chem import Draw, Descriptorsfrom collections import defaultdictfrom itertools import combinationsimport json, warnings, oswarnings.filterwarnings('ignore')# 从 .numbers 或 CSV 读取DATA_DIR = '.'CSV_FILE = os.path.join(DATA_DIR, 'clean_organolithium_unified_descriptors.csv')try:    import numbers_parser    doc = numbers_parser.Document(os.path.join(DATA_DIR, 'dataset-manual-corrected.numbers'))    table = doc.sheets[0].tables[0]    headers = [str(table.cell(0, c).value) for c in range(table.num_cols)]    rows = []    for r in range(1, table.num_rows):        row = {headers[c]: table.cell(r, c).value for c in range(table.num_cols)}        rows.append(row)    df = pd.DataFrame(rows)    print(f"从 .numbers 读取: {df.shape}")except:    df = pd.read_csv(CSV_FILE)    print(f"从 CSV 读取: {df.shape}")for col in ['tR1_s', 'T1_C', 'tR2_s', 'T2_C', 'yield_pct']:    df[col] = pd.to_numeric(df[col], errors='coerce')print(f"化合物数: {df['intermediate_smiles_canonical'].nunique()}")print(f"温度范围: {df['T1_C'].min():.0f} ~ {df['T1_C'].max():.0f} °C")print(f"类别分布:\n{df['intermediate_class'].value_counts()}")

## Step 2: 数据可视化 — yield vs tR 曲线竞争动力学模型: $\text{yield}(t_R) = y_{\max} \times (1 - e^{-k_f \cdot t_R}) \times e^{-k_d \cdot t_R}$

In [ ]:
# 选 3 个典型化合物examples = {    'p-NO2-PhLi (不稳定)': '[Li]c1ccc([N+](=O)[O-])cc1',    'tBu o-LiB (中等)': '[Li]c1ccccc1C(=O)OC(C)(C)C',    'p-Br-PhLi (稳定)': '[Li]c1ccc(Br)cc1',}methanol = df[df['electrophile'].str.contains('methanol', na=False, case=False)]fig, axes = plt.subplots(1, 3, figsize=(15, 4))for ax, (label, smi) in zip(axes, examples.items()):    sub = methanol[methanol['intermediate_smiles_canonical'] == smi]    if len(sub) == 0: sub = df[df['intermediate_smiles_canonical'] == smi]    temps = sorted(sub['T1_C'].dropna().unique())    clrs = plt.cm.coolwarm(np.linspace(0, 1, len(temps)))    for T, c in zip(temps, clrs):        t_data = sub[sub['T1_C'] == T].sort_values('tR1_s')        ax.scatter(t_data['tR1_s'], t_data['yield_pct'], c=[c], s=20, label=f'{T:.0f}°C')    ax.set_xscale('log'); ax.set_xlabel('tR (s)'); ax.set_ylabel('Yield (%)')    ax.set_title(label, fontsize=10); ax.legend(fontsize=7); ax.set_ylim(-5, 105)plt.tight_layout()plt.show()

## Step 3: 全局 Arrhenius 拟合 — 核心新方法**不再逐温度拟合**，而是每个化合物的所有温度数据**同时拟合 5 个参数**:$$\text{yield}(t_R, T) = y_{\max} \times (1 - e^{-A_f e^{-E_{a,f}/RT} \cdot t_R}) \times e^{-A_d e^{-E_{a,d}/RT} \cdot t_R}$$参数: $E_{a,f}, \ln A_f$ (生成) + $E_{a,d}, \ln A_d$ (分解) + $y_{\max}$

In [ ]:
R_gas = 8.314e-3  # kJ/(mol·K)def global_model(X, Ea_f, lnA_f, Ea_d, lnA_d, y_max):    """全局 Arrhenius 模型: 所有温度同时拟合"""    tR, T_K = X[0], X[1]    with np.errstate(over='ignore', under='ignore'):        k_f = np.exp(lnA_f - Ea_f / (R_gas * T_K))        k_d = np.exp(lnA_d - Ea_d / (R_gas * T_K))        return y_max * (1.0 - np.exp(-k_f * tR)) * np.exp(-k_d * tR)# 对每个化合物做全局拟合tr1 = df[df['tR_step'].str.startswith('tR1', na=False)].copy()global_results = []for smi in tr1['intermediate_smiles_canonical'].unique():    sub = tr1[tr1['intermediate_smiles_canonical'] == smi]    name = sub['intermediate'].iloc[0]        # 甲醇优先    methanol_sub = sub[sub['electrophile'].str.contains('methanol', na=False, case=False)]    if len(methanol_sub) >= 10: sub = methanol_sub        temps = sub['T1_C'].dropna().unique()    if len(temps) < 3: continue        tR_all = sub['tR1_s'].values.astype(float)    T_K_all = (sub['T1_C'].values.astype(float) + 273.15)    y_all = sub['yield_pct'].values.astype(float)    valid = np.isfinite(tR_all) & np.isfinite(T_K_all) & np.isfinite(y_all) & (tR_all > 0)    tR_all, T_K_all, y_all = tR_all[valid], T_K_all[valid], y_all[valid]    if len(tR_all) < 10: continue        X_data = np.array([tR_all, T_K_all])    try:        popt, _ = curve_fit(global_model, X_data, y_all,            p0=[30, 15, 40, 15, min(np.max(y_all)*1.05, 100)],            bounds=([0,-10,0,-10,10], [150,50,150,50,110]), maxfev=50000, method='trf')        Ea_f, lnA_f, Ea_d, lnA_d, y_max = popt        y_pred = global_model(X_data, *popt)        ss_res = np.sum((y_all-y_pred)**2); ss_tot = np.sum((y_all-np.mean(y_all))**2)        r2 = 1-ss_res/ss_tot if ss_tot > 0 else 0        if r2 > 0.3 and Ea_f > 0 and Ea_d > 0:            global_results.append({'intermediate':name, 'smi':smi, 'Ea_f':round(Ea_f,2),                'lnA_f':round(lnA_f,2), 'Ea_d':round(Ea_d,2), 'lnA_d':round(lnA_d,2),                'y_max':round(y_max,1), 'r2_global':round(r2,4), 'n_temps':len(temps)})    except: passglobal_df = pd.DataFrame(global_results).sort_values('Ea_d')print(f"全局拟合成功: {len(global_df)} 化合物")print(f"R² 中位数: {global_df['r2_global'].median():.3f}")print(f"Ea_f 范围: {global_df.Ea_f.min():.1f} ~ {global_df.Ea_f.max():.1f} kJ/mol")print(f"Ea_d 范围: {global_df.Ea_d.min():.1f} ~ {global_df.Ea_d.max():.1f} kJ/mol")global_df[['intermediate','Ea_f','Ea_d','lnA_f','lnA_d','r2_global']].head(10)

## Step 4: 从 5 参数计算任意温度的 t_max, t½, reactor有了 $E_{a,f}, \ln A_f, E_{a,d}, \ln A_d$ 后，在任意温度 T 可以算:- $k_f(T) = e^{\ln A_f - E_{a,f}/RT}$, $k_d(T) = e^{\ln A_d - E_{a,d}/RT}$  - $t_{\max} = \ln(k_f/k_d) / (k_f - k_d)$- $t_{1/2} = \ln 2 / k_d$- 反应器: flash ($t_{\max}<0.1$s) / flow ($0.1$-$60$s) / batch ($>60$s)

In [ ]:
def calc_tmax(Ea_f, lnA_f, Ea_d, lnA_d, T_C):    T_K = T_C + 273.15    k_f = np.exp(lnA_f - Ea_f/(R_gas*T_K))    k_d = np.exp(lnA_d - Ea_d/(R_gas*T_K))    if k_f <= k_d or k_f <= 0 or k_d <= 0: return 1e6    return np.log(k_f/k_d) / (k_f - k_d)def calc_thalf(Ea_d, lnA_d, T_C):    T_K = T_C + 273.15    k_d = np.exp(lnA_d - Ea_d/(R_gas*T_K))    return np.log(2)/k_d if k_d > 0 else 1e10def classify_reactor(t):    if t < 0.1: return 'flash'    elif t < 60: return 'flow'    else: return 'batch'def format_time(t):    if t >= 3600: return f"{t/3600:.1f}h"    elif t >= 60: return f"{t/60:.0f}min"    elif t >= 0.1: return f"{t:.2f}s"    else: return f"{t*1000:.1f}ms"# 展示每个化合物在 4 个温度的推荐print(f"{'化合物':25s} {'Ea_f':>5s} {'Ea_d':>5s} │ {'-78°C':>10s} {'-40°C':>10s} {'0°C':>10s} {'25°C':>10s}")print("="*85)for _, r in global_df.head(15).iterrows():    cols = []    for T in [-78, -40, 0, 25]:        tm = calc_tmax(r['Ea_f'], r['lnA_f'], r['Ea_d'], r['lnA_d'], T)        rc = classify_reactor(tm)        cols.append(f"{rc:>5s} {format_time(tm):>4s}")    print(f"{r['intermediate'][:25]:25s} {r['Ea_f']:5.1f} {r['Ea_d']:5.1f} │ {'  '.join(cols)}")

## Step 5: 结构分类 + 描述符准备基于 SMILES 结构（不是名字）正确分类:- **o-ArLi**: `[Li]c1ccccc1X` (1,2-位)- **m-ArLi**: `[Li]c1cccc(X)c1` (1,3-位)  - **p-ArLi**: `[Li]c1ccc(X)cc1` (1,4-位)- **oxiranylLi**: 含三元环氧

In [ ]:
def classify_structure(smi):    smi = str(smi)    if any(p in smi for p in ['CO1','C1CO1','OC1','C1OC1']): return 'oxiranylLi'    if '[Li]c1ccccc1' in smi and smi != '[Li]c1ccccc1': return 'o-ArLi'    if '[Li]c1ccc(' in smi: return 'p-ArLi'    if '[Li]c1cccc(' in smi: return 'm-ArLi'    if '[Li]c1' in smi: return 'hetero-ArLi'    return 'other'# 提取描述符 (已在原始数据集中)int_desc = {'q_C':'dft_charge_C_ipso','d_LiC':'dft_LiC_bond_A','BDE':'dft_LiC_BDE_kJ',    '%Vbur':'buried_vol_Li','Gsolv':'dft_Gsolv_kJ','HOMO':'dft_HOMO_eV',    'eta':'HOMO_LUMO_gap_eV','fukui':'fukui_f_minus_C',    'B1':'sterimol_B1','B5':'sterimol_B5','L':'sterimol_L',    'vol':'mol_volume','dipole':'dft_dipole_D'}unique = df.drop_duplicates(subset='intermediate_smiles_canonical')desc_table = unique[['intermediate','intermediate_smiles_canonical','intermediate_class']+list(int_desc.values())].copy()for col in int_desc.values():    desc_table[col] = pd.to_numeric(desc_table[col], errors='coerce')# 合并 global_df + 描述符 + 分类model_data = global_df.merge(desc_table, left_on='smi', right_on='intermediate_smiles_canonical', how='inner', suffixes=('','_d'))model_data['class'] = model_data['smi'].apply(classify_structure)model_data = model_data[model_data['r2_global'] > 0.6]  # 只用可靠拟合print(f"建模数据: {len(model_data)} 化合物")print(f"\n类别分布:\n{model_data['class'].value_counts()}")

## Step 6: 类别特异描述符筛选分类别穷举描述符组合，用 LOO-CV 评估对 Ea_f, Ea_d, lnA_f, lnA_d 4 个参数的预测能力。

In [ ]:
def loo_r2(X, y):    if len(X) < 5: return -999    yp = np.zeros_like(y, dtype=float)    for tr, te in LeaveOneOut().split(X):        yp[te] = LinearRegression().fit(X[tr],y[tr]).predict(X[te])    ss_r = np.sum((y-yp)**2); ss_t = np.sum((y-np.mean(y))**2)    return 1-ss_r/ss_t if ss_t>0 else 0desc_names = list(int_desc.keys())print(f"{'类别':>12s} {'参数':>6s} {'描述符':35s} {'LOO-R²':>8s} {'n':>4s}")print("="*70)class_best = {}  # 保存每个类别的最佳描述符for cls in ['oxiranylLi','m-ArLi','o-ArLi','p-ArLi']:    cls_data = model_data[model_data['class']==cls]    if len(cls_data) < 5: continue    class_best[cls] = {}        for target in ['Ea_f','Ea_d','lnA_f','lnA_d']:        best = {'r2':-999}        for np_ in [2, 3]:            for combo in combinations(desc_names, np_):                cols = [int_desc[d] for d in combo]                v = cls_data[cols+[target]].dropna()                if len(v) < max(4, len(cls_data)*0.5): continue                r2 = loo_r2(v[cols].values, v[target].values)                if r2 > best['r2']:                    best = {'r2':r2, 'descs':list(combo), 'cols':cols, 'n':len(v)}                if best['r2'] > -999:            class_best[cls][target] = best            star = ' ★' if best['r2'] > 0.5 else ''            print(f"{cls:>12s} {target:>6s} {'+'.join(best['descs']):35s} {best['r2']:8.3f} {best['n']:4d}{star}")    print()

## Step 7: 端到端 LOO 验证对每个化合物: LOO 预测 4 参数 → 计算 t_max(T) → 分类 flash/flow/batch → 和实际比较

In [ ]:
# LOO 端到端验证all_results = []for cls, config in class_best.items():    cls_data = model_data[model_data['class']==cls]    if len(cls_data) < 5: continue        for i in range(len(cls_data)):        test = cls_data.iloc[i]; train = cls_data.drop(cls_data.index[i])        pred = {}        for param in ['Ea_f','Ea_d','lnA_f','lnA_d']:            if param not in config:                pred[param] = train[param].mean(); continue            info = config[param]            v_train = train[info['cols']+[param]].dropna()            tv = [test[c] for c in info['cols']]            if any(np.isnan(v) for v in tv) or len(v_train) < 3:                pred[param] = train[param].mean()            else:                pred[param] = LinearRegression().fit(v_train[info['cols']].values, v_train[param].values).predict([tv])[0]                for T in [-78,-40,0,25]:            tm_a = calc_tmax(test['Ea_f'],test['lnA_f'],test['Ea_d'],test['lnA_d'],T)            tm_p = calc_tmax(pred['Ea_f'],pred['lnA_f'],pred['Ea_d'],pred['lnA_d'],T)            th_a = calc_thalf(test['Ea_d'],test['lnA_d'],T)            th_p = calc_thalf(pred['Ea_d'],pred['lnA_d'],T)            all_results.append({'name':test['intermediate'],'class':cls,'T':T,                'tm_a':tm_a,'tm_p':tm_p,'th_a':th_a,'th_p':th_p,                'r_a':classify_reactor(tm_a),'r_p':classify_reactor(tm_p)})rdf = pd.DataFrame(all_results)rdf['correct'] = rdf['r_a'] == rdf['r_p']# 按类别×温度print(f"{'类别':>12s} {'T':>6s} {'准确率':>8s}")print("="*30)for cls in ['oxiranylLi','m-ArLi','o-ArLi','p-ArLi']:    for T in [-78,-40,0,25]:        sub = rdf[(rdf['class']==cls)&(rdf['T']==T)]        if len(sub)==0: continue        print(f"{cls:>12s} {T:>6d}°C {sub['correct'].mean():>8.0%}")    print()print(f"总体: {rdf['correct'].mean():.1%} ({rdf['correct'].sum()}/{len(rdf)})")

## Step 8: 预测工具 — 输入 SMILES + 温度对已有化合物: 从全局拟合参数查表  对新化合物: 用类别特异描述符模型预测 4 参数

In [ ]:
def predict_compound(smiles, T_list=[-78,-40,0,25]):    cls = classify_structure(smiles)        # 查表: 已有化合物直接用全局拟合参数    match = global_df[global_df['smi']==smiles]    if len(match) > 0:        r = match.iloc[0]        params = {'Ea_f':r['Ea_f'],'lnA_f':r['lnA_f'],'Ea_d':r['Ea_d'],'lnA_d':r['lnA_d']}        source = '查表'    elif cls in class_best:        # 预测: 用描述符模型        m = desc_table[desc_table['intermediate_smiles_canonical']==smiles]        if len(m)==0: return f"SMILES 不在数据集中"        m = m.iloc[0]        cls_data = model_data[model_data['class']==cls]        params = {}        for param in ['Ea_f','Ea_d','lnA_f','lnA_d']:            if param not in class_best[cls]:                params[param] = cls_data[param].mean()            else:                info = class_best[cls][param]                vals = [pd.to_numeric(m.get(c), errors='coerce') for c in info['cols']]                if any(np.isnan(v) for v in vals):                    params[param] = cls_data[param].mean()                else:                    reg = LinearRegression().fit(cls_data[info['cols']+[param]].dropna()[info['cols']].values,                                                 cls_data[info['cols']+[param]].dropna()[param].values)                    params[param] = reg.predict([vals])[0]        source = '预测'    else:        return f"类别 {cls} 无模型"        print(f"  [{source}] {cls}  Ea_f={params['Ea_f']:.1f}  Ea_d={params['Ea_d']:.1f}")    for T in T_list:        tm = calc_tmax(params['Ea_f'],params['lnA_f'],params['Ea_d'],params['lnA_d'],T)        th = calc_thalf(params['Ea_d'],params['lnA_d'],T)        rc = classify_reactor(tm)        print(f"    T={T:>5d}°C  t_max={format_time(tm):>8s}  t½={format_time(th):>8s}  → {rc}")# 测试test_cases = [    ('[Li]c1cccc(C(=O)OC)c1', 'methyl m-lithiobenzoate'),    ('[Li]c1ccccc1C(=O)OC(C)(C)C', 'tBu o-lithiobenzoate'),    ('[Li]CCCC1CO1', '3-(oxiran-2-yl)propylLi'),    ('[Li]c1ccc(OC)cc1', 'p-Anisyllithium'),    ('[Li]c1ccc(C#N)cc1', 'p-cyanophenyllithium'),]for smi, name in test_cases:    print(f"\n{name} ({smi}):")    predict_compound(smi)

## Step 9: 总结### 方法全局 Arrhenius 拟合: 所有温度数据同时拟合 5 参数 (Ea_f, lnA_f, Ea_d, lnA_d, y_max)→ 类别特异描述符模型预测 4 参数 → 任意温度计算 t_max 和 t½ → 反应器推荐### 结果- 45 个化合物全局拟合 (R² 中位数 0.879)- 类别特异描述符: oxiranylLi 和 m-ArLi 4 参数全 R² > 0.68- 端到端反应器分类: 总体 80%, 0°C 95%### 参考文献- Ramachandran 2010, J. Phys. Chem. A — M06-2X 基准- Collum 2007, Angew. Chem. — 有机锂溶液动力学- Bannwarth 2019, J. Chem. Theory Comput. — GFN2-xTB- De Gennaro 2014, Lithium Compounds in Organic Synthesis — 数据来源